# Build RAG Model From Scratch

## Init

In [2]:
import os, sys

import numpy as np
from sentence_transformers import SentenceTransformer

## Pipeline

### Load Docs.

In [3]:
documents = [
    "Dogs are small domesticated mamals.",
    "Dogs are loyal animals often kept as pets.",
    "The Earth revolves around the Sun."
]

### Embedding

Convert documents into Vectors.

```math
f(\text{text}) = \text{vector}
```

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

d:\code\having_fun\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4160.82it/s]


Each of the objects Cats, Dogs, Earth are now have point in 384-dimensional space.

In [5]:
doc_embeddings = model.encode(documents)
print(type(doc_embeddings))
print(doc_embeddings.shape)

<class 'numpy.ndarray'>
(3, 384)


In [6]:
query = "Tell me about dogs"

query_embeddings = model.encode(query)

print(query_embeddings.shape)

(384,)


### Mesure Similarity

Cosine similarity

```math
\cos{\theta} = \frac{x.y}{|x| |y|}
```

Interpretation:
- 1 := identical direction
- 0 := unrelated
- -1 := opposite direction

In [7]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

Compare `query` against every doc in `documents`.

In [8]:
scores = []

for embedding in doc_embeddings:
    score = cosine_similarity(query_embeddings, embedding)
    scores.append(score)

print(scores)

[np.float32(0.5575728), np.float32(0.5765288), np.float32(0.08945567)]


### Retrieve Best Document

In [9]:
best_idx = np.argmax(scores)

best_doc = documents[best_idx]

print(best_doc)

Dogs are loyal animals often kept as pets.


### Retrieve Top-k

Get top 2

In [11]:
scores = np.array(scores)

top_idxs = np.argsort(scores)[-2:]

print(top_idxs)

[0 1]


Retrieve

In [12]:
for i in top_idxs:
    print(documents[i])

Dogs are small domesticated mamals.
Dogs are loyal animals often kept as pets.


### Generation

Everything so far is retrieval, means get the related documents and attach them to the prompt.
Now generate a human-like response.

In [14]:
context = "\n".join(documents[i] for i in reversed(top_idxs))

print(context)

Dogs are loyal animals often kept as pets.
Dogs are small domesticated mamals.


Construct prompt

In [15]:
prompt = f"""
Context:
{context}

Question:
{query}
"""

Now send the prompt to an LLM `llm` for the response.

```python
answer = llm(prompt)
```